# CKNNA Analysis: Cross-Model Representation Alignment

This notebook replaces the previous **label-alignment CKA** view with the
**Centered Kernel Nearest-Neighbor Alignment (CKNNA)** metric shown in the
benchmark excerpt. It computes **pairwise alignment between model
representations**, not model-vs-label similarity.

Default behavior:
- uses the union of ADMET benchmark SMILES as the shared molecule pool
- extracts one representation per model
- intersects models on shared successfully embedded SMILES
- computes **CKNNA with `k=5`**

Target model set:
- `minimol`
- `mole`
- `kpgt`
- `qip` (optional; skipped unless a local embedding payload exists)
- `pairmixer_4ds`


In [9]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
NOTEBOOK_DIR = ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if Path.cwd().resolve() != ROOT:
    os.chdir(ROOT)

import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pandas as pd

from cknna_utils import align_embeddings_on_shared_smiles, load_admet_smiles, load_model_embeddings, pairwise_cknna

matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 120})
sns.set_style("whitegrid")

K = 5
CACHE_DIR = ROOT / "results" / "cknna_model_alignment" / "cache"

MODEL_REGISTRY = {
    "minimol": {"kind": "minimol", "batch_size": 100},
    "mole": {"kind": "mole", "device": "cuda:0", "batch_size": 256},
    "kpgt": {
        "kind": "kpgt",
        "device": "cuda:0",
        "batch_size": 32,
        "kpgt_repo": "downloads/kpgt/upstream/KPGT",
        "checkpoint": "downloads/kpgt/base.pth",
    },
    "qip": {
        "kind": "qip",
        "embedding_path": "downloads/qip/embeddings.pt",
    },
    "pairmixer_4ds": {
        "kind": "pairmixer",
        "device": "auto",
        "batch_size": 32,
        "checkpoint": "models_checkpoints/toymix_dti_esmc_v3_litmolformer_v2_bbbc047/pairmixer_12M_ema/2026-05-05_18-24-47_20260505_182447/last.ckpt",
    },
}


2026-05-06 20:44:07.851357: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 20:44:07.899787: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-06 20:44:08.852384: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [10]:
smiles = load_admet_smiles()
print(f"Loaded {len(smiles):,} unique ADMET SMILES for alignment.")

model_embeddings, availability = load_model_embeddings(
    smiles,
    MODEL_REGISTRY,
    cache_dir=CACHE_DIR,
)
display(availability)

available = availability[availability["status"] == "ok"]["model"].tolist()
if len(available) < 2:
    raise RuntimeError("Need at least two available models to compute pairwise CKNNA.")

aligned = align_embeddings_on_shared_smiles(model_embeddings)
shared_n = next(iter(aligned.values())).shape[0]
print(f"Shared successfully embedded SMILES across available models: {shared_n:,}")
if shared_n <= K:
    raise RuntimeError(f"Need more than k={K} shared molecules, got {shared_n}.")


Found local copy...


Loaded 46,202 unique ADMET SMILES for alignment.
[kpgt] 46,202 unique SMILES (46,202 missing, 0 cached)
[kpgt] loading checkpoint from /home/shpark/prj-molrepr/graphium/downloads/kpgt/base.pth


/home/shpark/miniforge3/envs/kpgt/lib/python3.10/site-packages/torch/cuda/__init__.py:230: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(
/home/shpark/prj-molrepr/graphium/scripts/kpgt/kpgt_extract_embeddings.py:72: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for 

[kpgt] featurizing 46,202 SMILES…


kpgt forward:   0%|          | 0/46200 [00:00<?, ?mol/s]Traceback (most recent call last):
  File "/home/shpark/prj-molrepr/graphium/scripts/kpgt/kpgt_extract_embeddings.py", line 252, in <module>
    main()
  File "/home/shpark/prj-molrepr/graphium/scripts/kpgt/kpgt_extract_embeddings.py", line 230, in main
    embeddings = _batched_forward(
  File "/home/shpark/prj-molrepr/graphium/scripts/kpgt/kpgt_extract_embeddings.py", line 138, in _batched_forward
    batched = batched.to(device)
  File "/home/shpark/miniforge3/envs/kpgt/lib/python3.10/site-packages/dgl/heterograph.py", line 5714, in to
    ret._graph = self._graph.copy_to(utils.to_dgl_context(device))
  File "/home/shpark/miniforge3/envs/kpgt/lib/python3.10/site-packages/dgl/heterograph_index.py", line 255, in copy_to
    return _CAPI_DGLHeteroCopyTo(self, ctx.device_type, ctx.device_id)
  File "dgl/_ffi/_cython/./function.pxi", line 295, in dgl._ffi._cy3.core.FunctionBase.__call__
  File "dgl/_ffi/_cython/./function.pxi", line

...

kpgt forward:   0%|          | 0/46200 [00:02<?, ?mol/s]


pairmixer featurize:   0%|          | 0/46202 [00:00<?, ?it/s]

pairmixer forward:   0%|          | 0/1444 [00:00<?, ?it/s]

,model,kind,status,n_smiles,dim,note
0,minimol,minimol,missing,0,0,No module named 'scripts.minimol'
1,mole,mole,missing,0,0,No module named 'scripts.mole'
2,kpgt,kpgt,missing,0,0,Command '['/home/shpark/miniforge3/envs/kpgt/b...
3,qip,qip,missing,0,0,QIP embedding file not found: downloads/qip/em...
4,pairmixer_4ds,pairmixer,ok,46201,512,


RuntimeError: Need at least two available models to compute pairwise CKNNA.

In [ ]:
cknna_matrix = pairwise_cknna(aligned, k=K)
display(cknna_matrix.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cknna_matrix,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": f"CKNNA (k={K})"},
    ax=ax,
)
ax.set_title(f"Pairwise Representation Alignment via CKNNA (k={K})", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## Notes

- `qip` is treated as optional because no local repo asset was found during implementation.
- `pairmixer_4ds` defaults to the latest local 4-dataset PairMixer checkpoint present in `models_checkpoints/`.
- The helper caches embeddings under `results/cknna_model_alignment/cache/` so reruns avoid recomputing encoder outputs.
